# *<center> Trajectory Files: save once, analyse anywhere </center>*

### The following notebook works with SAVED ion flights — the `.traj.npz` files the GUI's *Save trajectories* button and `field_io.save_trajectories` write. Specific topics include:

* **Where these files live** and what is inside one.
* Loading a run back and plotting the paths — without re-flying anything.
* **Kinetic energy along each axis** ($x$, $y$, $z$) — derived exactly
  from the stored channels, which answers a natural worry: only the *net*
  KE is stored as a channel, but the full velocity vector is stored too,
  so the directional split is a projection, not a new simulation.
* Reading the same file with **no ion_gym at all** (numpy + json only).

### Conventions used in this document:

* **Units are mm, µs, mm/µs, eV, Da** — recorded in the file's own
  metadata, not assumed.
* **CAPITALS are parameters you may change.**
* A trajectory array has one row per recorded step and one column per
  channel; the column names are in the file's metadata and are never
  guessed by position.
* Directional kinetic energy is computed as
  $$\mathrm{KE}_i = \mathrm{KE} \cdot \frac{v_i^{2}}{|\mathbf{v}|^{2}}, \qquad i \in \lbrace x, y, z \rbrace,$$
  which uses only stored quantities — no mass, no constants, no
  re-run — and sums back to the stored KE by construction.

____

## Stage 0 — imports

## The instrument, before any statistics

The device this notebook flies by default, drawn from the **solver's own
electrode mask** (not a redrawing) with example ion paths exactly as
flown. You are looking at a tapered stack of RF ring electrodes: the
ring bores shrink toward the exit, adjacent rings carry opposite RF
phases (the effective-potential wall that keeps ions off the metal), and
a DC gradient walks the ions along the axis through buffer gas. The
paths converge radially as the taper narrows — the funnel doing its job:
accepting a wide, diffuse cloud and delivering a thin beam to the exit.

Deck: `examples/ion_funnel_rz.json` (the default — Stage A lets you
point `SPEC_PATH` at any example, and the stages below adapt to whatever
you load). A geometry figure is not decoration — if the picture and the
solved model can disagree, every number below is unverifiable.

In [ ]:
from ion_gym.viz.nb_panels import show_instrument_panel
# banked panel when present, else rendered live from the deck
# of record (self-contained on a public checkout).
show_instrument_panel('examples/ion_funnel_rz.json', banked='panel_funnel.png', height=520)


In [ ]:
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# path anchor (2026-08-08): every relative path below is REPO-ROOT-relative,
# independent of where Jupyter/VS Code set the working directory.
from ion_gym.io.paths import repo_root as _repo_root
import os as _os
_os.chdir(_repo_root())

import os
import io
import numpy as np
import panel as pn
import plotly.graph_objects as go
from IPython.display import display, Markdown

pn.extension()

# ion_gym imports are used ONLY to (a) print the server-side save
# directory and (b) fly the optional demo file when you have no file yet.
# The ANALYSIS below never touches them — Stage B's reader is numpy+json.
from ion_gym.io.sim_spec import SimSpec
from ion_gym.io import field_io
from ion_gym.io.paths import fields_dir
from ion_gym.physics.sim_build import build_run
from ion_gym.physics.ensemble_driver import IonResult

# ---- FIGURE SIZE ----------------------------------------------------------
FIGSIZE = (940, 420)     # pixels; pass figsize= per-call to override one

In [ ]:
# ---- solve-cache control ------------------------------------------
# Banked field bases make re-runs fast but can serve STALE fields
# after a geometry edit. Set True to clear the cache and force fresh
# solves for this run.
CLEAR_FIELD_CACHE = False
if CLEAR_FIELD_CACHE:
    import shutil
    from pathlib import Path
    from ion_gym.io.fa_cache import DEFAULT_ROOT
    _cache = Path(DEFAULT_ROOT)
    if 'ion_gym' not in _cache.name:
        raise ValueError(f'refusing to clear {_cache} — not an ion_gym '
                         'cache dir (check ION_GYM_CACHE)')
    if _cache.exists():
        shutil.rmtree(_cache)
        print(f'cleared solve cache: {_cache}')
    else:
        print(f'solve cache already empty: {_cache}')


## Stage A — where trajectory files live (and the two buttons)

The GUI has **two** ways to get trajectories out, and they write
*different files to different places*:

| button | where the file lands | ion members | metadata |
|---|---|---|---|
| **Export run (NPZ, compact)** | your browser's download folder | `ion_0000, ion_0001, …` | `columns` + a numeric summary table (no `_meta`) |
| **Save trajectories** | the server-side directory printed below | `t0, t1, …` | full `_meta` JSON (columns, units, per-ion m/z, label, geometry key) |

They are the *same trajectories* in two containers, and both now carry
the column names as **plain npz arrays** — so the reader below needs no
JSON and no spec; it detects which kind it was handed from the members
alone. Export goes through the browser so you can see it land; Save
writes where the GUI's *Load as run* button reads overlays back from.

Every save — from the GUI button or from code — lands in **one declared
directory**, `ion_gym.io.paths.fields_dir()`, unless you pass an explicit
path. The cell prints that directory and lists what is in it, so "where
did it go?" has a one-line answer. (The GUI's save message also shows the
full absolute path.)

In [ ]:
print("server-side save directory (the *Save trajectories* button):")
print("   ", os.path.abspath(fields_dir()))
existing = sorted(str(p) for p in fields_dir().glob("*.traj.npz"))
if existing:
    print(f"\n{len(existing)} trajectory file(s) found:")
    for p in existing:
        meta = field_io.read_trajectory_meta(p)
        print(f"   {os.path.basename(p)}  — {meta.get('label')!r}, "
              f"{len(meta.get('ion_index', []))} ions")
else:
    print("\n(no .traj.npz files yet — the next cell creates one to work "
          "with, exactly as the GUI's Save button would)")

# ---- deck-inherited physics: visible and overridable -----------------
# The deck loaded below supplies the drive, the ion, the gas and the
# integration settings. Leave an entry None to INHERIT it from the deck;
# set one to OVERRIDE. Whatever ends up in force is printed, so this
# notebook's output always states its own operating point.
# (One namespaced dict, not loose globals: the first version used bare
# names like KE_EV and N_IONS, which collided with the parameters these
# notebooks already own -- and silently changed them.)
DECK_OVERRIDES = dict(
    rf_v=None, rf_f=None,        # confining-drive amplitude (V) / freq (Hz)
    mz_list=None, charge=None,   # e.g. [622.0] / 1
    ke_ev=None,                  # (lo, hi) eV
    source_t_k=None,             # K, thermal spread of initial velocities
    n_ions=None,                 # ions flown
    gas_on=None, gas=None,       # True/False / e.g. "N2"
    p_torr=None, gas_t_k=None,   # buffer-gas pressure (Torr) / temp (K)
    dt_ns=None, t_max_us=None,   # integration step (ns) / flight time (us)
)


In [ ]:
from ion_gym.io.deck_params import describe_deck
# Repo-relative root: every path in this cell derives from it, so the
# notebook runs on any machine and in any cell order.
from pathlib import Path as _P
from ion_gym.io.paths import repo_root as _rr
ROOT = _P(_rr())
# ---- Parameters of the demo run -- change to suit your system ------------
SPEC_PATH = str(ROOT / 'examples/ion_funnel_rz.json')
N_IONS = 10

# Fly a small ensemble and SAVE it — the same call the GUI button makes —
# so the rest of the notebook has a real file to load. Skipped if you set
# TRAJ_PATH below to a file you already have.
TRAJ_PATH = None      # or a path to your own .traj.npz

if TRAJ_PATH is None:
    demo_spec = SimSpec.from_json(SPEC_PATH)
    # Say what this deck supplied -- drives, ion, gas, integration --
    # so the operating point is visible where the spec is built.
    describe_deck(demo_spec)
    demo_spec.source.n_ions = N_IONS
    model, fly, columns, births = build_run(demo_spec)
    results = []
    for i in range(len(births)):
        traj_i, summary_i = fly(i)
        results.append(IonResult(index=i, traj=traj_i, summary=summary_i))
    TRAJ_PATH = field_io.save_trajectories(results, demo_spec,
                                           label="notebook 05 demo")
print("working with:", os.path.abspath(TRAJ_PATH))


## Stage B — pick a file (type a path *or* use the loader)

Two ways to choose the `.npz`, whichever suits you: paste a path into the
text box, or use the file loader to upload one (an *Export run* download,
for instance). Whichever you fill in wins; if you fill neither, the demo
file from the previous cell is used, so the notebook still runs end to
end. Run this cell, make your choice in the widgets, then run the rest.

In [ ]:
# type a path here …
traj_path_input = pn.widgets.TextInput(
    name="trajectory file path",
    placeholder="/full/path/to/run.npz  (Export or Save file)",
    width=560)
# … or load a file here (upload; e.g. an 'Export run (NPZ, compact)' download)
traj_file_input = pn.widgets.FileInput(accept=".npz")
pn.Column(
    pn.pane.Markdown("**Option 1 — path**"), traj_path_input,
    pn.pane.Markdown("**Option 2 — load a .npz**"), traj_file_input)

In [ ]:
# Resolve the chosen source: an uploaded file (bytes) beats a typed path,
# a typed path beats the demo fallback. read_trajectory_file (next cell)
# accepts either a path string OR an in-memory file, so both routes feed
# the same reader.
def chosen_source():
    if traj_file_input.value:                 # uploaded bytes
        name = traj_file_input.filename or "uploaded.npz"
        print(f"using uploaded file: {name}")
        return io.BytesIO(traj_file_input.value)
    if traj_path_input.value.strip():         # typed path
        p = traj_path_input.value.strip()
        print(f"using path: {os.path.abspath(p)}")
        return p
    print(f"using demo file: {os.path.abspath(TRAJ_PATH)}")
    return TRAJ_PATH

SOURCE = chosen_source()

## Stage B2 — one reader, both file kinds, no spec, no JSON

The function below is the **only loader this notebook uses**. It reads the
npz **array members alone** — no JSON decode, no spec, no geometry, no
ion_gym import — and returns the trajectories with a plain metadata dict,
whichever button wrote the file. It accepts a path or an uploaded file
object. The columns are the contract: everything downstream reads
channels by NAME, never by position.

In [ ]:
# ---- THE reader: npz members ONLY - no json, no spec, no ion_gym ---------
import re
import numpy as np

def read_trajectory_file(source):
    # Load either trajectory container from npz ARRAY MEMBERS ALONE:
    #   * Save-trajectories file: ion members t0,t1,... plus plain arrays
    #     columns, label, mz_da, summary, summary_cols (a _meta blob may
    #     also be present; this reader ignores it - no json).
    #   * Export-run (NPZ, compact) download: ion members ion_0000,...
    #     plus columns + summary + summary_cols.
    # `source` is a path string OR an open/in-memory file object, so a
    # typed path and an uploaded file feed the same code. Returns
    # (trajectories, meta): a list of (rows x columns) arrays in ion order
    # and a plain dict (columns, summaries, mz_da, label, source_format).
    # A file with no plain 'columns' member is refused with its member
    # list - a wrong file gets a diagnosis, not a guess.
    npz = np.load(source, allow_pickle=False)
    names = set(npz.files)
    if "columns" not in names:
        raise ValueError(
            "not a readable trajectory npz: no plain 'columns' member "
            f"(members: {sorted(names)}). Files saved before 2026-07-26 "
            "kept columns only inside the json _meta; re-save from the "
            "current GUI to get the plain 'columns' member.")
    columns = [str(c) for c in npz["columns"]]
    ion_keys = sorted((k for k in names if re.fullmatch(r"(t|ion_)\d+", k)),
                      key=lambda k: int(re.sub(r"\D", "", k)))
    trajectories = [npz[k] for k in ion_keys]
    summaries = []
    if "summary" in names and "summary_cols" in names:
        scols = [str(c) for c in npz["summary_cols"]]
        for row in npz["summary"]:
            summaries.append({c: float(v) for c, v in zip(scols, row)})
    is_save = any(re.fullmatch(r"t\d+", k) for k in names)
    meta = {"columns": columns, "summaries": summaries,
            "mz_da": ([float(x) for x in npz["mz_da"]]
                      if "mz_da" in names else None),
            "label": (str(npz["label"][0]) if "label" in names
                      else "(no label member)"),
            "source_format": ("save_trajectories" if is_save
                              else "export_run (compact NPZ)")}
    return trajectories, meta


trajectories, meta = read_trajectory_file(SOURCE)
columns = list(meta["columns"])
print("format      :", meta["source_format"])
print("label       :", meta.get("label"))
print("columns     :", columns)
print("ions loaded :", len(trajectories))
print("m/z (Da)    :", meta.get("mz_da") or "(no mz_da member in this file)")
fates = [s.get("fate", s.get("kind")) for s in meta.get("summaries", [])]
print("fates       :", fates if fates else "(none recorded)")

## Stage C — plot the paths

The rows below are the stored recording — nothing is re-flown. (If you
want a saved run back *inside the GUI* for overlay, that is the *Load as
run* button, which uses `field_io.load_trajectories` and additionally
checks the file's geometry key against the loaded spec, warning on a
mismatch. This notebook needs none of that: the file is self-sufficient.)

In [ ]:
i_t = columns.index("t")
i_x, i_y = columns.index("x"), columns.index("y")

fig_paths = go.Figure()
for k, traj in enumerate(trajectories):
    fig_paths.add_scatter(x=traj[:, i_x], y=traj[:, i_y], mode="lines",
                          line=dict(width=1.2), name=f"ion {k}",
                          showlegend=(k < 6))
fig_paths.update_layout(
    title=f"{meta.get('label')} — {len(trajectories)} saved paths "
          "(no re-flight: these are the stored rows)",
    xaxis_title="x (mm)", yaxis_title="y (mm)",
    width=FIGSIZE[0], height=FIGSIZE[1],
    margin=dict(l=70, r=20, t=52, b=52))
fig_paths

## Stage D — kinetic energy along each axis

The recorded channels include the net kinetic energy `ke_ev` **and** the
velocity components `vx, vy, vz` (with `speed` = $|\mathbf{v}|$). Kinetic
energy is quadratic in velocity, so it splits over axes exactly:

**Eqn #1**

$$\mathrm{KE} = \tfrac{1}{2} m |\mathbf{v}|^2
   = \tfrac{1}{2} m v_x^2 + \tfrac{1}{2} m v_y^2 + \tfrac{1}{2} m v_z^2
   = \mathrm{KE}_x + \mathrm{KE}_y + \mathrm{KE}_z.$$

Dividing through by $\tfrac{1}{2} m |\mathbf{v}|^2$ gives the working
form: each axis's share is $v_i^2 / |\mathbf{v}|^2$ of the stored KE. No
mass enters, so this works even on files saved before m/z was recorded —
and the three parts sum back to the stored channel, which the cell checks
rather than asserts.

**Why this split is physically interesting here:** in an RF device the
transverse components carry the micromotion (they oscillate at the drive
frequency), while the axial component carries the transport. The net KE
mixes those; the split separates them.

In [ ]:
# ---- Parameters of the KE analysis -- change to suit your system ---------
ION_TO_SHOW = 0       # which saved ion the per-axis trace plots
MZ_DA = None          # ion mass (Da) for ABSOLUTE KE when the file stores
                      # neither ke_ev nor mz_da; None => read mz_da if
                      # present, else fall back to MZ_FALLBACK_DA below.
MZ_FALLBACK_DA = 300.0

KG_AMU = 1.66053906660e-27    # kg per Da
E_CHG = 1.602176634e-19       # C
MMUS_TO_MS = 1.0e3            # mm/us -> m/s

# The ONLY channels this analysis truly needs are vx, vy, vz. speed and
# ke_ev are conveniences: if the recording did not store them (e.g. you
# recorded the field components e_x/e_y/e_z instead), they are DERIVED
# here from the velocity — the per-axis split is exact either way.
i_vx, i_vy, i_vz = (columns.index("vx"), columns.index("vy"),
                    columns.index("vz"))
have_speed = "speed" in columns
have_ke = "ke_ev" in columns
i_speed = columns.index("speed") if have_speed else None
i_ke = columns.index("ke_ev") if have_ke else None

# resolve the mass used for absolute KE when ke_ev is not stored
if MZ_DA is not None:
    m_da = float(MZ_DA)
elif meta.get("mz_da"):
    vals = [x for x in meta["mz_da"] if x is not None and np.isfinite(x)]
    m_da = float(vals[0]) if vals else MZ_FALLBACK_DA
else:
    m_da = MZ_FALLBACK_DA
if not (have_ke):
    _mz_src = ("from mz_da" if meta.get("mz_da")
               else "MZ_FALLBACK_DA — set MZ_DA if this is wrong")
    print(f"note: this file has no 'ke_ev' channel; absolute KE is computed "
          f"from vx,vy,vz using m = {m_da:g} Da ({_mz_src}).")
if not have_speed:
    print("note: no 'speed' channel; deriving it from vx,vy,vz.")

def _speed2(traj):
    if have_speed:
        return traj[:, i_speed] ** 2
    return (traj[:, i_vx] ** 2 + traj[:, i_vy] ** 2 + traj[:, i_vz] ** 2)

def _ke_total(traj):
    # eV. Use the stored channel if present, else 1/2 m v^2 / e.
    if have_ke:
        return traj[:, i_ke]
    v2_ms = _speed2(traj) * (MMUS_TO_MS ** 2)      # (m/s)^2
    return 0.5 * (m_da * KG_AMU) * v2_ms / E_CHG

def ke_by_axis(traj):
    # (ke_x, ke_y, ke_z) per recorded step. KE_i = KE * v_i^2 / |v|^2 — the
    # split needs only the velocity components; the total KE is stored or
    # derived above. Rows with |v| = 0 get zero on every axis.
    speed2 = _speed2(traj)
    ke = _ke_total(traj)
    with np.errstate(invalid="ignore", divide="ignore"):
        fx = np.where(speed2 > 0, traj[:, i_vx] ** 2 / speed2, 0.0)
        fy = np.where(speed2 > 0, traj[:, i_vy] ** 2 / speed2, 0.0)
        fz = np.where(speed2 > 0, traj[:, i_vz] ** 2 / speed2, 0.0)
    return ke * fx, ke * fy, ke * fz

traj = trajectories[ION_TO_SHOW]
ke_x, ke_y, ke_z = ke_by_axis(traj)
ke_tot = _ke_total(traj)
closure = np.max(np.abs((ke_x + ke_y + ke_z) - ke_tot))
denom = ke_tot.mean() if ke_tot.mean() != 0 else np.nan
print(f"ion {ION_TO_SHOW}: {len(traj)} recorded steps")
print(f"per-axis KE sums back to the total KE to within "
      f"{closure:.2e} eV (float roundoff) — the split is exact")
print(f"time-averaged shares: x {ke_x.mean()/denom:.0%}, "
      f"y {ke_y.mean()/denom:.0%}, z {ke_z.mean()/denom:.0%}")

In [ ]:
fig_ke = go.Figure()
x_axis = traj[:, i_x]
for name, series, color in (("KE_x (axial)", ke_x, "#00e5ff"),
                            ("KE_y", ke_y, "#ff7f0e"),
                            ("KE_z", ke_z, "#d62728"),
                            ("KE total", _ke_total(traj),
                             "#aaaaaa")):
    fig_ke.add_scatter(x=x_axis, y=series, mode="lines", name=name,
                       line=dict(width=1.6 if "total" not in name else 1.0,
                                 color=color,
                                 dash="dot" if "total" in name else None))
fig_ke.update_layout(
    title=f"kinetic energy by axis vs distance — ion {ION_TO_SHOW} "
          f"(derived from stored vx, vy, vz; nothing re-flown)",
    xaxis_title="axial position x (mm)",
    yaxis_title="kinetic energy (eV)",
    width=FIGSIZE[0], height=FIGSIZE[1],
    margin=dict(l=70, r=20, t=52, b=52))
fig_ke

### CSV export (optional)
Write any dict of equal-length 1-D arrays (the per-axis KE traces
below, or any channel selection) to CSV for external tools. Helper per
recorded per saved ion.

In [ ]:
import csv
import numpy as np
def dict_to_csv(d, path):
    """Write a dict of equal-length 1-D arrays to CSV, keys as the header."""
    cols = {k: np.asarray(v).ravel() for k, v in d.items()}
    lengths = {k: v.size for k, v in cols.items()}
    if len(set(lengths.values())) != 1:
        raise ValueError(f"columns have unequal lengths: {lengths}")
    n = next(iter(lengths.values()))
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(cols.keys())                       # header: x, y, z
        w.writerows(zip(*(cols[k] for k in cols)))    # one row per step
    print(f"wrote {n} rows x {len(cols)} cols -> {path}")

# per-axis KE of the ion selected above (ION_TO_SHOW), one row per step
ke_axes = {"t_us": traj[:, i_t], "x_mm": traj[:, i_x],
           "ke_x_ev": ke_x, "ke_y_ev": ke_y, "ke_z_ev": ke_z}
# Generated files go to the ONE outputs home, not the working directory:
# writing to the CWD left a copy wherever the notebook happened to be run.
from ion_gym.io import paths as _paths
dict_to_csv(ke_axes, str(_paths.outputs_dir("ke_axes.csv")))


Read the traces against the physics: the dotted total rides on top of the
axial component through the transport sections, while the transverse
components spike where the RF micromotion is strongest — near the ring
apertures — and collapse on the axis. If a study needs the transverse
temperature (an effective-temperature or heating question), this split of
the *saved* file is the measurement; nothing needed re-flying.

The ensemble version is one loop away — the cell below averages each
axis's share over every saved ion, in axial bins.

In [ ]:
# ---- Parameters of the ensemble profile -- change to suit your system ----
N_BINS = 30

edges = None
share_sums, ke_sums, counts = None, None, None
for traj_k in trajectories:
    kx, ky, kz = ke_by_axis(traj_k)
    if edges is None:
        lo = min(float(t[:, i_x].min()) for t in trajectories)
        hi = max(float(t[:, i_x].max()) for t in trajectories)
        edges = np.linspace(lo, hi, N_BINS + 1)
        share_sums = np.zeros((3, N_BINS))
        ke_sums = np.zeros(N_BINS)
        counts = np.zeros(N_BINS)
    which = np.clip(np.digitize(traj_k[:, i_x], edges) - 1, 0, N_BINS - 1)
    for b in range(N_BINS):
        sel = which == b
        if not sel.any():
            continue
        share_sums[0, b] += kx[sel].sum()
        share_sums[1, b] += ky[sel].sum()
        share_sums[2, b] += kz[sel].sum()
        ke_sums[b] += _ke_total(traj_k)[sel].sum()
        counts[b] += sel.sum()

centres = 0.5 * (edges[:-1] + edges[1:])
ok = counts > 20                     # sparse bins are OMITTED, visibly
print(f"{int(ok.sum())} of {N_BINS} bins had enough samples; the rest are "
      "omitted rather than plotted as noise")
fig_ens = go.Figure()
for j, (name, color) in enumerate((("KE_x", "#00e5ff"),
                                   ("KE_y", "#ff7f0e"),
                                   ("KE_z", "#d62728"))):
    fig_ens.add_scatter(x=centres[ok], y=share_sums[j, ok] / counts[ok],
                        mode="lines+markers", name=name,
                        line=dict(color=color))
fig_ens.update_layout(
    title=f"ensemble mean KE by axis vs distance "
          f"({len(trajectories)} saved ions)",
    xaxis_title="axial position x (mm)",
    yaxis_title="mean kinetic energy (eV)",
    width=FIGSIZE[0], height=FIGSIZE[1],
    margin=dict(l=70, r=20, t=52, b=52))
fig_ens

## Stage E — per-axis KE distributions, with an effective-temperature fit

Histogram each axis's kinetic energy on a **log count** scale. For a single
velocity component in thermal equilibrium the distribution is
Maxwell-Boltzmann on one degree of freedom,
**Eqn #2**

$$p(\mathrm{KE}_i) \propto \mathrm{KE}_i^{-1/2}\,
   e^{-\mathrm{KE}_i / k_B T},$$
so on a log-count axis the tail is a **straight line** whose slope is
$-1/(k_B T)$. Fitting that slope is a direct read of the effective
temperature along that axis — the line you wanted, and the physics it
encodes.

Two knobs:

* **`X_UNITS`** — `"eV"` plots energy directly; `"kT"` divides the axis by
  $k_B T_\mathrm{ref}$ (a reference temperature you set), so the x-axis
  is in units of $k_B T$ and a thermal tail has slope $-1$.
* **`FIT_FROM_PCTL`** — the fit ignores the low-energy rise (the
  $\mathrm{KE}^{-1/2}$ part and binning noise near zero) and fits only the
  populated tail above this percentile.

Each panel prints two temperatures: **`T_slope`** from the log-linear fit,
and **`T_mean`** $= 2\langle \mathrm{KE}_i\rangle / k_B$ (the per-degree-of-
freedom mean). They agree only when the axis is genuinely thermal; the
printed **$R^2$** says how straight the tail actually is. In a driven,
collisional device they will differ — the axial axis carries directed
transport, not just random thermal motion — and that disagreement is
information, not error.

In [ ]:
# ---- Parameters of the KE-distribution fit -- change to suit your system -
X_UNITS = "eV"        # "eV" or "kT"
T_REF_K = 300.0       # reference T for the kT axis (only used if X_UNITS=="kT")
N_BINS_HIST = 100
FIT_FROM_PCTL = 20    # fit the tail above this percentile of each axis's KE

KB_EV = 8.617333262e-5    # Boltzmann constant, eV/K

# pool every ion's per-axis KE (the ensemble distribution)
KEX, KEY, KEZ = [], [], []
for traj_k in trajectories:
    kx, ky, kz = ke_by_axis(traj_k)
    KEX.append(kx)
    KEY.append(ky)
    KEZ.append(kz)
ke_axes = {"x": np.concatenate(KEX), "y": np.concatenate(KEY),
           "z": np.concatenate(KEZ)}

def fit_log_tail(ke_vals, n_bins, from_pctl):
    # Histogram; fit ln(counts) vs KE over the populated tail. Returns
    # (centers, counts, T_slope_K, r2, line_x, line_y) with line_* the
    # fitted straight line in the SAME (count, eV) space for overlay.
    # An axis with NO energy (e.g. ke_z when the solve is 2-D and vz==0)
    # has an empty ke_pos: percentile/mean on it would raise, so return
    # an empty-but-valid result rather than crash.
    ke_pos = ke_vals[ke_vals > 0]
    if ke_pos.size == 0:
        return (np.zeros(0), np.zeros(0, int), np.nan, np.nan, None, None)
    counts, edges = np.histogram(ke_pos, bins=n_bins)
    centers = 0.5 * (edges[:-1] + edges[1:])
    tail = (counts > 0) & (centers > np.percentile(ke_pos, from_pctl))
    if tail.sum() < 2:
        return centers, counts, np.nan, np.nan, None, None
    A = np.vstack([centers[tail], np.ones(tail.sum())]).T
    (slope, intercept), *_ = np.linalg.lstsq(A, np.log(counts[tail]),
                                             rcond=None)
    pred = A @ [slope, intercept]
    ss_res = float(((np.log(counts[tail]) - pred) ** 2).sum())
    ss_tot = float(((np.log(counts[tail])
                     - np.log(counts[tail]).mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    T_slope = -1.0 / (slope * KB_EV) if slope < 0 else np.nan
    line_x = centers[tail]
    line_y = np.exp(pred)
    return centers, counts, T_slope, r2, line_x, line_y

# unit scaling for the x-axis
scale = 1.0 if X_UNITS == "eV" else 1.0 / (KB_EV * T_REF_K)
x_label = "kinetic energy (eV)" if X_UNITS == "eV" else           f"kinetic energy (k_B T, T_ref = {T_REF_K:g} K)"

from plotly.subplots import make_subplots
fig_hist = make_subplots(rows=1, cols=3,
                         subplot_titles=("KE_x", "KE_y", "KE_z"))
colors = {"x": "#1f77b4", "y": "#2ca02c", "z": "#d62728"}
for col, (ax, ke_vals) in enumerate(ke_axes.items(), start=1):
    ke_pos = ke_vals[ke_vals > 0]
    if ke_pos.size == 0:
        # No energy on this axis — a 2-D solve leaves ke_z identically zero.
        # Say so plainly on the panel and in the print; do NOT invent a fit.
        print(f"KE_{ax}: no energy on this axis "
              f"(all zero — expected for the out-of-plane axis of a 2-D "
              f"solve); no temperature to fit")
        fig_hist.add_annotation(
            text=f"no {ax}-energy<br>(2-D solve: v{ax} = 0)",
            showarrow=False, xref=f"x{col}", yref=f"y{col}",
            x=0.5, y=0.5, font=dict(size=13, color="#888"),
            row=1, col=col)
        fig_hist.update_xaxes(title_text=x_label, row=1, col=col)
        fig_hist.update_yaxes(title_text="count" if col == 1 else None,
                              row=1, col=col)
        continue
    centers, counts, T_slope, r2, lx, ly = fit_log_tail(
        ke_vals, N_BINS_HIST, FIT_FROM_PCTL)
    T_mean = 2.0 * ke_pos.mean() / KB_EV
    fig_hist.add_bar(x=centers * scale, y=counts, marker_color=colors[ax],
                     opacity=0.65, showlegend=False, row=1, col=col)
    if lx is not None:
        fig_hist.add_scatter(x=lx * scale, y=ly, mode="lines",
                             line=dict(color="black", width=2, dash="dash"),
                             showlegend=False, row=1, col=col)
    print(f"KE_{ax}: T_slope = {T_slope:7.0f} K   "
          f"T_mean = {T_mean:7.0f} K   R^2(tail) = {r2:.3f}")
    fig_hist.update_xaxes(title_text=x_label, row=1, col=col)
    fig_hist.update_yaxes(type="log", title_text="count" if col == 1 else None,
                          row=1, col=col)
fig_hist.update_layout(
    title=f"per-axis KE distributions (log count) with log-linear tail fit "
          f"— x in {X_UNITS}",
    width=FIGSIZE[0] + 120, height=FIGSIZE[1],
    bargap=0, margin=dict(l=70, r=20, t=60, b=52))
fig_hist

## Stage E2 — separating thermal energy from RF micromotion

The fit in Stage E gives a temperature *per axis*, but on an RF-confined
axis the "temperature" from a single Maxwellian is contaminated two ways,
and this stage corrects both:

1. **Bulk drift.** $T_\mathrm{mean} = 2\langle \mathrm{KE}_i\rangle/k_B$
   counts directed motion as though it were thermal. The drift-subtracted temperature uses
   the *variance* of the velocity,
   $$k_B T_i^{\mathrm{var}} = m\,\mathrm{Var}(v_i)
     = m\big(\langle v_i^2\rangle - \langle v_i\rangle^2\big),$$
   which equals $T_\mathrm{mean}$ only when the axis has zero mean
   velocity.
2. **RF micromotion.** The confining field drives a coherent oscillation
   at the RF period — energy that is not thermal. Averaging each ion's
   velocity over whole RF periods gives the **secular** (drift/thermal)
   motion; the residual is the **micromotion**. Their variances give a
   secular temperature and a micromotion "temperature" separately, so the
   two contributions are no longer conflated in one number.

This needs the ion mass (from `mz_da`) and the RF period. If the file has
no `mz_da` (an older Export file), the cell says so and reports only the
shape-based split it can still compute.

In [ ]:
# ---- Parameters of the thermal/micromotion split ------------------------
RF_PERIOD_US = None   # None => read from the file's drive metadata if present,
                      # else set the RF period (us) by hand, e.g. 1.25

KB_J = 1.380649e-23           # J/K
KG_AMU = 1.66053906660e-27    # kg per Da
MMUS_TO_MS = 1.0e3            # mm/us -> m/s

# mass: from the plain mz_da member (per ion); fall back to a single value.
mz_list = meta.get("mz_da")
if mz_list and all(x is not None and np.isfinite(x) for x in mz_list):
    m_kg_per_ion = [float(x) * KG_AMU for x in mz_list]
else:
    m_kg_per_ion = None
    print("no per-ion m/z in this file — variance/secular temperatures need "
          "mass; skipping the K-valued split and showing velocity variance "
          "ratios only.")

# RF period: from the drive table if the file carries one, else the param.
period_us = RF_PERIOD_US
if period_us is None:
    fr = None
    for s in meta.get("summaries", []):
        pass
    # trajectory files don't store the drive; use RF_PERIOD_US or infer from
    # the dominant oscillation in vy via its autocorrelation first zero.
    if period_us is None:
        print("RF_PERIOD_US not set and not in file; estimating it from the "
              "velocity autocorrelation.")
def infer_period_us(t, v):
    # crude but robust: FFT the detrended velocity, take the dominant freq.
    if len(t) < 8:
        return None
    dt = np.median(np.diff(t))
    vv = v - v.mean()
    freqs = np.fft.rfftfreq(len(vv), d=dt)
    amp = np.abs(np.fft.rfft(vv))
    amp[0] = 0.0
    fk = freqs[np.argmax(amp)]
    return (1.0 / fk) if fk > 0 else None

In [ ]:
def var_temperature(v_mmus, m_kg):
    # drift-subtracted: T = m*Var(v)/kB, v converted mm/us -> m/s
    v = np.asarray(v_mmus) * MMUS_TO_MS
    return m_kg * np.var(v) / KB_J

def secular_micro(t_us, v_mmus, period_us):
    # per-window mean velocity = secular; residual = micromotion.
    t = np.asarray(t_us)
    if len(t) < 4 or not period_us:
        return None, None
    win = np.floor((t - t[0]) / period_us).astype(int)
    sec = np.empty_like(v_mmus, dtype=float)
    for w in np.unique(win):
        sel = win == w
        sec[sel] = np.asarray(v_mmus)[sel].mean()
    return sec, np.asarray(v_mmus) - sec

i_t = columns.index("t")
per_axis = {}
for ax, ivx in (("x", columns.index("vx")),
                ("y", columns.index("vy")),
                ("z", columns.index("vz"))):
    sec_v, mic_v, all_v, masses = [], [], [], []
    for k, traj_k in enumerate(trajectories):
        t = traj_k[:, i_t]
        v = traj_k[:, ivx]
        p = period_us or infer_period_us(t, v)
        sc, mc = secular_micro(t, v, p)
        all_v.append(v)
        if sc is not None:
            sec_v.append(sc)
            mic_v.append(mc)
        if m_kg_per_ion is not None:
            masses.append(m_kg_per_ion[k] if k < len(m_kg_per_ion)
                          else m_kg_per_ion[0])
    all_v = np.concatenate(all_v)
    if np.allclose(all_v, 0.0):
        print(f"{ax}: no motion on this axis (2-D solve) — skipped")
        continue
    m_kg = masses[0] if masses else None
    if m_kg is not None:
        T_var = var_temperature(all_v, m_kg)
        line = f"{ax}: T_var(drift-subtracted) = {T_var:7.1f} K"
        if sec_v:
            T_sec = var_temperature(np.concatenate(sec_v), m_kg)
            T_mic = var_temperature(np.concatenate(mic_v), m_kg)
            line += (f"   secular = {T_sec:7.1f} K   "
                     f"micromotion = {T_mic:7.1f} K")
        print(line)
    else:
        # no mass: report the variance ratio secular:micro (dimensionless)
        if sec_v:
            rv = np.var(np.concatenate(mic_v)) / max(np.var(all_v), 1e-30)
            print(f"{ax}: micromotion fraction of velocity variance = {rv:.2f}")
    per_axis[ax] = all_v

The secular temperature is the one to quote as *the* temperature of the
trapped cloud along each axis; the micromotion figure is coherent drive
energy, not random thermal energy. A large micromotion value on the
RF-confined axes is expected and does not mean the cloud is thermally
hot — it is the coherent energy of the driven oscillation, the price of
confinement. If secular and micromotion are comparable on an axis you
did not expect to be driven, that is worth a second look at the geometry
or the drive assignment.

## Stage E3 — the two estimators, side by side, with fitted curves

This puts both temperature methods on one figure so you can *see* where
they agree and where they part.

**Top row — velocity space.** Each axis's velocity histogram with a
Gaussian overlaid at the measured mean and variance. This is the
variance/moment method: $k_B T_i = m\,\mathrm{Var}(v_i)$. The Gaussian
hugs the data by construction when the motion is thermal, and its
*offset from zero* is the bulk drift — the thing that inflates a
naive $T_\mathrm{mean}$.

**Bottom row — energy space.** Each axis's KE histogram (log count) with
the 1-D Maxwell energy distribution
$p(E)\propto E^{-1/2}e^{-E/k_BT}$ overlaid at **two** temperatures: the
drift-subtracted $T_\mathrm{var}$ (solid) and the tail-slope
$T_\mathrm{slope}$ (dashed). Where the two curves separate, and where
each departs from the bars, is the visual statement of how much the axis
is *not* a single drift-free Maxwellian.

Read together: a thermal, drift-free axis shows a centred Gaussian on top
and both energy curves hugging the bars below, all three temperatures
agreeing. A driven or drifting axis shows an offset Gaussian, energy
curves that split, and three different numbers — and the variance-based
$T_\mathrm{var}$ from the top row is the one to trust.

In [ ]:
# ---- Stage E3 parameters -----------------------------------------------
# The two-estimator temperature physics + figure now live in the
# FRAMEWORK (ion_gym.physics.traj_stats + ion_gym.viz.viz_core), so this
# notebook and the GUI share ONE implementation. FIT_FROM_PCTL is the
# exposed tail-fit parameter (the GUI binds a widget to it): for a
# non-thermal distribution T_slope depends on where the tail fit starts,
# so it is stated, not buried.
FIT_FROM_PCTL_E3 = 20    # tail percentile for the slope fit
N_BINS_E3 = 80

from ion_gym.physics.traj_stats import (two_estimator_report,
                                        KG_AMU)
from ion_gym.viz.viz_core import temperature_estimator_figure

# mass (Da): reuse m_da from the KE cell if defined, else file metadata.
try:
    m_da_e3 = float(m_da)
except NameError:
    _mz = meta.get('mz_da')
    _ok = [x for x in (_mz or []) if x is not None and np.isfinite(x)]
    m_da_e3 = float(_ok[0]) if _ok else 300.0
    print(f"using m = {m_da_e3:g} Da "
          f"({'from mz_da' if _ok else 'fallback — set m_da_e3'})")
m_kg_e3 = m_da_e3 * KG_AMU

# assemble ensemble per-axis velocity and KE arrays from the loaded file
i_vx3, i_vy3, i_vz3 = (columns.index('vx'), columns.index('vy'),
                       columns.index('vz'))
KX, KY, KZ = [], [], []
for tk in trajectories:
    kx, ky, kz = ke_by_axis(tk)
    KX.append(kx)
    KY.append(ky)
    KZ.append(kz)
V_axes = {'x': np.concatenate([t[:, i_vx3] for t in trajectories]),
          'y': np.concatenate([t[:, i_vy3] for t in trajectories]),
          'z': np.concatenate([t[:, i_vz3] for t in trajectories])}
KE_axes = {'x': np.concatenate(KX), 'y': np.concatenate(KY),
           'z': np.concatenate(KZ)}


In [ ]:
# per-axis two-estimator temperatures + the committed figure — ONE call
rep = two_estimator_report(V_axes, KE_axes, m_kg_e3,
                           fit_from_pctl=FIT_FROM_PCTL_E3,
                           n_bins=N_BINS_E3)
hdr = f"{'axis':<5}{'T_var(K)':>11}{'T_mean(K)':>11}{'T_slope(K)':>12}"
hdr += f"{'mean v(mm/us)':>15}{'drift?':>9}"
print(hdr)
print('-' * len(hdr))
print('(T_var and T_mean AGREE on a drift-free axis; they SPLIT only\n'
      ' under bulk drift. T_slope reads the tail independently, so\n'
      ' T_var != T_slope flags a non-thermal distribution.)')
for ax in ('x', 'y', 'z'):
    e = rep.get(ax)
    if e is None:
        print(f"{ax:<5}{'--':>11}{'--':>11}{'--':>12}{'--':>15}{'--':>9}")
        continue
    print(f"{ax:<5}{e['T_var']:>11.0f}{e['T_mean']:>11.0f}"
          f"{e['T_slope']:>12.0f}{e['v_mean']:>15.4f}"
          f"{('yes' if e['drift'] else 'no'):>9}")

temperature_estimator_figure(V_axes, KE_axes, m_kg_e3,
                             fit_from_pctl=FIT_FROM_PCTL_E3,
                             n_bins=N_BINS_E3)


____

## Portability notes

The reader in Stage B already IS the no-dependency version — numpy + json
only, no spec, no ion_gym — so copy that one function anywhere. The cell
below re-runs it on this notebook's file purely as the round-trip check.

In [ ]:
# ---- round-trip check: the Stage B reader, re-imported cold --------------
# Re-read the chosen file and recompute KE_x on the SAME ion Stage D used.
# Uses the SAME derivation as Stage D (speed/KE from vx,vy,vz when the
# convenience channels are absent), so a file recording e_x/e_y/e_z instead
# of speed/ke_ev round-trips too. float32-appropriate RELATIVE tolerance.
trajectories_check, meta_check = read_trajectory_file(SOURCE)
tr = trajectories_check[ION_TO_SHOW]
cc = list(meta_check["columns"])
jvx, jvy, jvz = cc.index("vx"), cc.index("vy"), cc.index("vz")
s2 = tr[:, jvx] ** 2 + tr[:, jvy] ** 2 + tr[:, jvz] ** 2
if "ke_ev" in cc:
    ke_tot_chk = tr[:, cc.index("ke_ev")]
else:
    ke_tot_chk = 0.5 * (m_da * KG_AMU) * (s2 * (MMUS_TO_MS ** 2)) / E_CHG
ke_x_check = np.where(s2 > 0, ke_tot_chk * tr[:, jvx] ** 2 / s2, 0.0)
m_check = float(ke_x_check.mean())
m_stage_d = float(ke_x.mean())
rel = abs(m_check - m_stage_d) / max(abs(m_stage_d), 1e-12)
print(f"round-trip KE_x mean: {m_check:.6f} eV "
      f"(Stage D computed: {m_stage_d:.6f} eV; relative diff {rel:.1e})")
assert rel < 1e-4, (
    f"round-trip mismatch: {m_check} vs {m_stage_d} (rel {rel:.2e}). "
    "If you changed ION_TO_SHOW between cells, re-run Stage D first.")

Same rules as the field export for other environments: MATLAB reads the
unzipped `.npy` members with `npy-matlab` and `jsondecode(char(...))` for
`_meta`; Julia uses `NPZ.jl`; and `unzip file.traj.npz` yields plain
`.npy` files any reader loads.

### Notes

* **Per-axis KE needs nothing recorded beyond the defaults** — `vx, vy,
  vz, speed, ke_ev` are standard channels, and the split is exact.
* **m/z is in the metadata** (`mz_da`, one entry per saved ion) for
  analyses that need absolute momenta; older files lack
  it, and the loader says so rather than guessing.
* The GUI's *Load trajectories* button reads these same files back as
  stored runs for overlay — this notebook and the GUI share one loader.

## Read-out

- **Trajectories are data, not pictures.** Saving the flown arrays (with named channels and their units) means an analysis can be redone, checked, or re-plotted years later without re-flying — which matters because a re-flight with a different step size or seed is a *different experiment*.
- **Name channels; never index them.** Positional access to a trajectory array breaks silently the moment a channel is added — the same class of defect that broke every planar flight in this toolkit once. The named-column views exist to make that impossible.